In [0]:
--1.1 monthly traffic performance
select
    year,
    month,
    year_month,
    count(total_sessions) as observed_days,
    sum(total_sessions) as monthly_sessions,
    round(avg(total_sessions), 0) as avg_daily_sessions
from workspace.redfin_gold.daily_traffic
where day <= date('2019-07-30')
    and is_missing_date = false
    and is_future_date = false
group by 1,2,3
order by 1,2 

Databricks visualization. Run in Databricks to view.

In [0]:
--1.1a  pivot  - avg daily session 
with avg_session as
(select
    year,
    month,
    year_month,
    count(total_sessions) as observed_days,
    sum(total_sessions) as monthly_sessions,
    round(avg(total_sessions), 0) as avg_daily_sessions
from workspace.redfin_gold.daily_traffic
where day <= date('2019-07-30')
    and is_missing_date = false
    and is_future_date = false
group by 1,2,3
) 

select month, 
       sum(case when year =2018 then monthly_sessions end ) as monthly_sessions_2018,
       sum(case when year =2019 then monthly_sessions end ) as monthly_sessions_2019, 
       sum(case when year =2018 then avg_daily_sessions end ) as avg_daily_sessions_2018,
       sum(case when year =2019 then avg_daily_sessions end ) as avg_daily_sessions_2019
from avg_session 
group by 1 order by 1



Databricks visualization. Run in Databricks to view.

In [0]:
-- 1.2 YOY growth  jan - july 
with monthly_traffic as (
    select
        year,
        month,
        avg(total_sessions) as avg_daily_sessions
    from workspace.redfin_gold.daily_traffic
    where year in (2018, 2019)
        and month between 1 and 7
        and is_missing_date = false
        and is_future_date = false
    group by 1,2 
      
)
select
    month,
    round(max(case when year = 2018 then avg_daily_sessions end), 0) as avg_daily_sessions_2018,
    round(max(case when year = 2019 then avg_daily_sessions end), 0) as avg_daily_sessions_2019,
    round((  max(case when year = 2019 then avg_daily_sessions end)
            / max(case when year = 2018 then avg_daily_sessions end) - 1) * 100,1) as yoy_growth_pct
from monthly_traffic
group by 1 order by 1 

Databricks visualization. Run in Databricks to view.

In [0]:
-- q1.3 seasonality  -- if remove sep 

with monthly_traffic as (
    select
        month(to_date(year_month)) as month,
        count(*) as observed_days,
        sum(total_sessions) as monthly_sessions,
        avg(total_sessions) as avg_daily_sessions
    from workspace.redfin_gold.daily_traffic
    where year = 2018
      and is_missing_date = false
      and is_future_date = false
      and month(to_date(year_month)) != 9
    group by month(to_date(year_month))
)
,annual_average as (
    select
        sum(monthly_sessions) / sum(observed_days) as annual_avg_daily_sessions
    from monthly_traffic
)

select
    m.month,
    m.observed_days,
    round(m.avg_daily_sessions, 0) as avg_daily_sessions,
    round(a.annual_avg_daily_sessions, 0) as annual_avg_daily_sessions,
    round( m.avg_daily_sessions / a.annual_avg_daily_sessions,3) as seasonal_index,
    round((m.avg_daily_sessions / a.annual_avg_daily_sessions - 1) * 100,1) as seasonal_adjustment_pct
from monthly_traffic m
cross join annual_average a
order by m.month;


Databricks visualization. Run in Databricks to view.

In [0]:
-- q1.3 seasonality --include 9 

with monthly_traffic as (
    select
        month(to_date(year_month)) as month,
        count(*) as observed_days,
        sum(total_sessions) as monthly_sessions,
        avg(total_sessions) as avg_daily_sessions
    from workspace.redfin_gold.daily_traffic
    where year = 2018
      and is_missing_date = false
      and is_future_date = false
    group by month(to_date(year_month))
)

,annual_average as
 (    select
        sum(monthly_sessions) / sum(observed_days) as annual_avg_daily_sessions
    from monthly_traffic
)

select
    m.month,
    m.observed_days,
    round(m.avg_daily_sessions, 0) as avg_daily_sessions,
    round(a.annual_avg_daily_sessions, 0) as annual_avg_daily_sessions,
    round(   m.avg_daily_sessions / a.annual_avg_daily_sessions, 3) as seasonal_index,
    round( (m.avg_daily_sessions / a.annual_avg_daily_sessions - 1) * 100,1) as seasonal_adjustment_pct
from monthly_traffic m
cross join annual_average a
order by m.month;


Databricks visualization. Run in Databricks to view.

In [0]:
--2.1 channel/medium mix  % 
with monthly_channel_sessions as 
(
    select
        year,
        month,
        year_month,
        medium,
        sum(sessions) as channel_sessions
    from workspace.redfin_gold.fact_daily_traffic
    where is_missing_date = false
        and is_future_date = false
    group by 1,2,3,4 
)

, base as (
select
    year_month,
    medium,
    round( channel_sessions * 1.0 / sum(channel_sessions) over (partition by year_month),3) as traffic_share_pct
from monthly_channel_sessions) 

---pivot  email, direct, organic, push, share, referal, paid , other 
select year_month, 
      sum(case when medium = 'email' then traffic_share_pct else 0 end) as email,
      sum(case when medium = 'direct' then traffic_share_pct else 0 end) as direct,
      sum(case when medium = 'organic' then traffic_share_pct else 0 end) as organic,   
      sum(case when medium = 'push' then traffic_share_pct else 0 end) as push,
      sum(case when medium = 'share' then traffic_share_pct else 0 end) as share,
      sum(case when medium = 'referal' then traffic_share_pct else 0 end) as referal,
      sum(case when medium = 'paid' then traffic_share_pct else 0 end) as paid,
      sum(case when medium = 'other' then traffic_share_pct else 0 end) as other 
from base 
group by 1 
order by 1 


Databricks visualization. Run in Databricks to view.

In [0]:
--2.2 avg daily by month by medium   -- volume 
with observed_dates as 
(select
        day,
        year,
        month,
        year_month
    from workspace.redfin_gold.daily_traffic
    where is_missing_date = false
        and is_future_date = false
)

,media as 
(select distinct
        medium
    from workspace.redfin_gold.fact_daily_traffic
    where medium is not null
)

,daily_channel_sessions as 
(select
        day,
        medium,
        sum(sessions) as daily_sessions
    from workspace.redfin_gold.fact_daily_traffic
    where is_missing_date = false
        and is_future_date = false
    group by 1, 2 
)

,daily_channel_spine as 

( select
        d.day,
        d.year,
        d.month,
        d.year_month,
        m.medium,
        coalesce(s.daily_sessions, 0) as daily_sessions
    from observed_dates d
    cross join media m
    left join daily_channel_sessions s 
    on d.day = s.day and m.medium = s.medium
)

select  year_month,
        medium,
        count(*) as observed_days,
        round(avg(daily_sessions), 0) as avg_daily_sessions
from daily_channel_spine
group by 1,2 
order by 1,2 

Databricks visualization. Run in Databricks to view.

In [0]:
--Q2.3 medium growth and contribution to overall traffic growth. jan to july 
with period_channel_sessions as (
    select
        year,
        medium,
        sum(sessions) as total_sessions
    from workspace.redfin_gold.fact_daily_traffic
    where year in (2018, 2019)
        and is_missing_date = false
        and is_future_date = false
        and (
            (medium = 'share' and month between 5 and 7)
            or (medium <> 'share' and month between 1 and 7)
        )
    group by
        year,
        medium
),
period_days as (
    select
        year,
        case
            when year = 2018 then 212
            else 211
        end as observed_days
    from workspace.redfin_gold.daily_traffic
    where year in (2018, 2019)
        and month between 1 and 7
        and is_missing_date = false
        and is_future_date = false
    group by year
),
share_days as (
    select
        year,
        count(*) as observed_days
    from workspace.redfin_gold.daily_traffic
    where year in (2018, 2019)
        and month between 5 and 7
        and is_missing_date = false
        and is_future_date = false
    group by year
),
average_daily_sessions as (
    select
        p.year,
        p.medium,
        p.total_sessions /
            case
                when p.medium = 'share' then s.observed_days
                else d.observed_days
            end as avg_daily_sessions
    from period_channel_sessions p
    left join period_days d
        on p.year = d.year
    left join share_days s
        on p.year = s.year
),
channel_comparison as (
    select
        medium,
        max(case when year = 2018 then avg_daily_sessions end)
            as avg_daily_sessions_2018,
        max(case when year = 2019 then avg_daily_sessions end)
            as avg_daily_sessions_2019
    from average_daily_sessions
    group by medium
),
channel_growth as (
    select
        medium,
        avg_daily_sessions_2018,
        avg_daily_sessions_2019,
        avg_daily_sessions_2019 - avg_daily_sessions_2018
            as incremental_daily_sessions
    from channel_comparison
)
select
    medium,
    round(avg_daily_sessions_2018, 0) as avg_daily_sessions_2018,
    round(avg_daily_sessions_2019, 0) as avg_daily_sessions_2019,
    round(incremental_daily_sessions, 0) as incremental_daily_sessions,
    round(
        incremental_daily_sessions * 1.0
        / avg_daily_sessions_2018,
        3
    ) as yoy_growth_pct
from channel_growth
order by incremental_daily_sessions desc;